**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Diffusion II: Score Matching & SDEs

The rigorous sequel [Diffusion Models](./Diffusion_Models.ipynb) gestured at: what the network *actually* learns is the **score** $\nabla_x \log p(x)$, the discrete chain is an [SDE](../Intro_Math/Stochastic_Processes/Stochastic_Processes_2.ipynb) in disguise, and generation is that SDE run backwards. Verified on a Gaussian mixture where the true score is available in closed form — the oracle most tutorials never check.

## 1. Pre-requisites

[Diffusion Models](./Diffusion_Models.ipynb) (the practical loop), [Stochastic Processes II](../Intro_Math/Stochastic_Processes/Stochastic_Processes_2.ipynb) S4 (Brownian motion, Itô).

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
torch.manual_seed(0); rng = np.random.default_rng(0)

# ground-truth distribution: a 2-D Gaussian mixture — score known in CLOSED FORM
centers = torch.tensor([[-2.0, 0.0], [2.0, 0.0], [0.0, 2.2]])
sig2 = 0.35**2
def sample_data(n):
    idx = torch.randint(0, 3, (n,))
    return centers[idx] + 0.35*torch.randn(n, 2)
def true_score(x, t_var=0.0):
    """∇ log p_t for the mixture convolved with N(0, t_var) — exact."""
    v = sig2 + t_var
    d2 = ((x[:, None, :] - centers[None])**2).sum(-1)
    w = torch.softmax(-d2/(2*v), dim=1)
    mu_post = (w[:, :, None] * centers[None]).sum(1)
    return (mu_post - x) / v

**Why a toy mixture.** Three Gaussian blobs are not a demo of what diffusion can *do* — for that, see the images in [Diffusion Models](./Diffusion_Models.ipynb). They are here because a mixture of Gaussians is one of the few distributions whose score we can write down exactly, in the `true_score` function above. Everything in this workshop is a claim of the form "the network learns the score" or "the chain is an SDE," and on this toy problem each claim becomes a number we can check rather than a sentence we have to trust. That is the trade we are making: we give up visual impressiveness and buy an oracle.

---
### 🕐 Session 1 of 3 — *The Score Function* (~40 min)
**Goal:** learn ∇ log p by denoising; verify against the closed-form mixture score.
**Builds on:** [Diffusion Models](./Diffusion_Models.ipynb). &nbsp; **Feeds into:** Session 2 (Langevin & SDEs).

---

## 2. The Gradient of the Log-Density

💡 **Intuition.** The score $s(x) = \nabla_x \log p(x)$ is a *compass field*: at every point it points toward higher probability. You never need the (intractable) normalizing constant — gradients of $\log p$ kill it. And the miracle that makes it learnable: **denoising score matching** — training a network to predict the noise added to data is, up to scale, training it to output the score of the *noised* distribution ($s = -\varepsilon/\sigma$). Diffusion I's 'predict the noise' loss was secretly score estimation all along. Here we can *prove* it: our mixture's score has a closed form to compare against.

In [ ]:
# train a denoiser at ONE noise level; compare its implied score with the exact one
# ORACLE: implied score −net(x)/σ vs the closed-form mixture score at this noise level

# YOUR CODE HERE


**What just happened.** Median cosine similarity **0.9987**, median relative error **0.098** — the red arrows sit almost exactly on the grey ones. That is the whole claim of this session, measured: the network was trained *only* to predict the noise $\varepsilon$, and never saw `true_score` during training, yet $-\text{net}(x)/\sigma$ reproduces the closed-form score of the noised mixture. "Predict the noise" and "estimate the score" are the same objective wearing different clothes.

Two details worth reading off the plot. The arrows agree tightly wherever the blue data points are, and drift apart in the far corners — the network only ever saw training samples where the data lives, so it has no reason to be right in the tails. And the direction is far more accurate than the magnitude (cosine 0.999 against a ~10% relative error), which is exactly what we want, because the samplers in Sessions 2 and 3 use the score as a *direction to move* and then control the step size themselves.

---
### 🕐 Session 2 of 3 — *Langevin Dynamics & the Forward SDE* (~40 min)
**Goal:** climb the score with noise: Langevin sampling; the diffusion chain as an SDE.
**Builds on:** Session 1; [Stochastic Processes II](../Intro_Math/Stochastic_Processes/Stochastic_Processes_2.ipynb). &nbsp; **Feeds into:** Session 3 (the reverse SDE & probability flow).

---

## 3. Sampling = Noisy Gradient Ascent

💡 **Intuition.** Given a score, **Langevin dynamics** samples: $x \mathrel{+}= \frac{\eta}{2} s(x) + \sqrt{\eta}\, \xi$ — climb the compass field, but inject just enough noise that you *explore* the distribution instead of collapsing to its modes ([SGD's noise](../Intro_Math/Optimization/Optimization.ipynb), now load-bearing). The catch that motivated diffusion: with far-apart modes, plain Langevin mixes badly — which is why diffusion runs a *family* of scores across noise levels: high noise merges the modes for easy travel, low noise sharpens the details. And in the continuum, Diffusion I's chain **is** the SDE $dx = -\tfrac12\beta x\, dt + \sqrt{\beta}\, dB$ — an Ornstein–Uhlenbeck process whose marginals we can check exactly.

In [ ]:
# ORACLE: the forward SDE's variance must follow the OU closed form

# YOUR CODE HERE


**What just happened.** We never solved the SDE — we simulated it, one small Euler–Maruyama step at a time, exactly as Diffusion I's loop does. Itô's calculus says the variance of that process must follow $\mathrm{Var}(t) = e^{-\beta t}\,\mathrm{Var}(0) + (1 - e^{-\beta t})$, and the simulated dots track that closed form to a maximum gap of **0.030**. The chain really is an Ornstein–Uhlenbeck process.

Read the shape, not just the error: the curve starts at the data's own variance and relaxes to 1, no matter what we started with. That limit is why the reverse process in Session 3 can begin from `torch.randn` — the forward SDE deliberately forgets the data and converges to a standard Gaussian we know how to sample. The residual scatter around the black line is Euler–Maruyama discretization error plus finite-sample noise from 4000 particles; it shrinks with smaller `dt`, and it is the same error the reverse sampler will pay.

---
### 🕐 Session 3 of 3 — *The Reverse SDE & Probability Flow* (~40 min)
**Goal:** run time backwards with the score; sample the mixture and audit mode weights.
**Builds on:** Session 2.

---

## 4. Anderson's Time Machine

💡 **Intuition.** The stunning theorem (Anderson, 1982): the forward SDE has an exact **reverse**: $dx = [-\tfrac12\beta x - \beta\, s_t(x)]\, dt + \sqrt{\beta}\, d\bar{B}$ — identical dynamics plus a score-guided drift. Everything unknown about 'undoing noise' is packed into $s_t$, the exact object Session 1 taught us to learn. Drop the noise term and halve the score drift and you get the **probability-flow ODE** — deterministic, same marginals, the bridge to flow matching and fast samplers (DDIM is its discretization).

In [ ]:
# train a time-conditional score net, then sample by reverse SDE — audit the result
# reverse SDE from pure noise
# ORACLE: mode weights should be ≈ 1/3 each; mode means ≈ the centers

# YOUR CODE HERE


**What just happened.** We started from pure Gaussian noise — no data whatsoever — and ran Anderson's reverse SDE for 400 steps using nothing but the learned score. Out came the mixture: weights **0.354 / 0.330 / 0.316** against a true 1/3 each, with recovered centers $(-2.08, 0.05)$, $(2.06, -0.02)$, $(0.03, 2.19)$ against the true $(-2, 0)$, $(2, 0)$, $(0, 2.2)$.

The weights matter more than the centers here, and it is worth saying why. Getting the centers right only shows the sampler finds the modes — a mode-seeking method that ignored the noise term would manage that too. Getting the *weights* right shows it visits them in the correct proportion, which is the difference between a sampler and an optimizer. That is the property Session 2's noise term buys, now measured end to end.

Note also what this run did *not* require: no Metropolis correction, no annealing schedule to tune by hand, no separate model per noise level. One time-conditional score network, plus a theorem from 1982, and generation falls out. The residual few percent in the weights is the accumulation of everything we have already accounted for — score error in the tails from Session 1, discretization error from Session 2, and finite-sample noise from 6000 points.

## 5. Conclusion

'Predict the noise' is score estimation (verified against a closed-form score, cosine ≈ 1); the chain is an OU SDE (variance audited against Itô); and Anderson's reverse SDE turns the learned compass into a sampler whose mode weights and centers match the truth. Diffusion I's recipe now has its complete mathematical spine.

---
## Where next

- [Stochastic Processes II](../Intro_Math/Stochastic_Processes/Stochastic_Processes_2.ipynb) — the Itô calculus underneath.
- [Optimal Transport](./Optimal_Transport.ipynb) — probability flow as a transport map; flow matching lives here.